# Glaciers — catalog & behaviour (no network)

This notebook explores the `glaciers` **catalog** and the backend's **behaviour** without downloading any data. You'll learn the five datasets, how a bounding box maps to a GTN-G region, why the output shape is *per instance*, and why `aggregate=` is rejected.

## The catalog

`Catalog` loads the bundled dataset definitions. `available()` lists the shipped dataset ids across the three sources.

In [1]:
from earthlens.glaciers import Catalog

cat = Catalog()
cat.available()

2026-06-28 17:54:18 | INFO | pyramids.base.config | Logging is configured.


['glims:outlines',
 'rgi:outlines',
 'wgms:front_variation',
 'wgms:mass_balance',
 'wgms:state']

Each id resolves to a typed row carrying its source and **output kind** — `vector` for the RGI/GLIMS outlines, `tabular` for the WGMS fluctuations. This is the field the backend copies onto its `OUTPUT_KIND` per instance.

In [2]:
import pandas as pd

rows = [
    {'id': d, 'source': cat.get(d).source,
     'output_kind': cat.get(d).output_kind,
     'long_name': cat.get(d).long_name}
    for d in cat.available()
]
pd.DataFrame(rows)

,id,source,output_kind,long_name
0,glims:outlines,glims,vector,"GLIMS time-series glacier outlines (WFS, bbox ..."
1,rgi:outlines,rgi,vector,RGI 7.0 glacier outlines (per GTN-G region)
2,wgms:front_variation,wgms,tabular,WGMS glacier front variation (length change)
3,wgms:mass_balance,wgms,tabular,WGMS glacier mass balance (winter / summer / a...
4,wgms:state,wgms,tabular,WGMS glacier state (elevation / area / length ...


## The GTN-G region table

RGI ships one shapefile per **GTN-G first-order region**. The catalog carries each region's bounding box(es) and download URL, and the backend uses them to turn your bbox into the region(s) to fetch. Region 10 (North Asia) crosses the antimeridian, so it has two boxes.

In [3]:
region_rows = [
    {'id': r, 'name': cat.regions[r].name,
     'n_boxes': len(cat.regions[r].bboxes)}
    for r in sorted(cat.regions)
]
pd.DataFrame(region_rows)

,id,name,n_boxes
0,01,Alaska,1
1,02,Western Canada and USA,1
2,03,Arctic Canada North,1
3,04,Arctic Canada South,1
4,05,Greenland Periphery,1
5,06,Iceland,1
6,07,Svalbard and Jan Mayen,1
7,08,Scandinavia,1
8,09,Russian Arctic,1
9,10,North Asia,2


`regions_for_bbox` is the helper that maps an area of interest (`[west, south, east, north]`) to the overlapping region id(s). An Alpine box lands in region 11; an East-Siberian box lands in region 10 via its second sub-box.

In [4]:
from earthlens.glaciers import regions_for_bbox

alps = regions_for_bbox([6.8, 45.8, 7.2, 46.05], cat.regions)
siberia = regions_for_bbox([150.0, 60.0, 160.0, 65.0], cat.regions)
alps, siberia

(['11'], ['10'])

## Per-instance output shape

Because the catalog mixes vector and tabular datasets, the backend sets `OUTPUT_KIND` from the dataset you choose. A construction with an RGI dataset is `vector`; one with a WGMS dataset is `tabular`. (We construct the backends here — no download happens until `.download()`.)

In [5]:
from earthlens import EarthLens

rgi = EarthLens(data_source='glaciers', variables=['rgi:outlines'],
                region='11')
wgms = EarthLens(data_source='glaciers',
                 variables=['wgms:mass_balance'])
rgi.datasource.OUTPUT_KIND, wgms.datasource.OUTPUT_KIND

2026-06-28 17:54:19.802 | INFO     | earthlens.earthlens:__init__:1046 - No `path` given; download() writes 'glaciers' output under earthlens-data\glaciers/ (load() uses a temp dir).


2026-06-28 17:54:19.806 | INFO     | earthlens.earthlens:__init__:1046 - No `path` given; download() writes 'glaciers' output under earthlens-data\glaciers/ (load() uses a temp dir).


('vector', 'tabular')

## Why `aggregate=` is rejected

Outlines and fluctuations are pre-computed inventories / measurements, not gridded fields, so there is no meaningful time-window or spatial reduction. Passing `aggregate=` raises `NotImplementedError` — a clear signal rather than a silent no-op.

In [6]:
try:
    rgi.datasource.download(aggregate=object())
except NotImplementedError as exc:
    print('rejected:', str(exc).split(':')[0])

rejected: Glaciers.download(aggregate=...) is not supported


## Takeaway

The catalog tells you the source, the output shape, and (for RGI) the region layout before you download a byte. Continue with [GLIMS & WGMS](03_glims_and_wgms.ipynb) for the time-series outlines and the mass-balance record.